# Kvasir-SEG dataset prep: images + masks + manifests

Downloads Kvasir-SEG from Hugging Face, creates train/val/test splits, saves images and masks locally, and writes metadata/manifests under `out/`.

TODO:: This dataset is not found on Hugging face, manual download didn't work either, need to come back to it

In [1]:
import os
from pathlib import Path
import random
import math

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from datasets import load_dataset


In [2]:
# Config
# Hugging Face dataset; using a working public mirror
HF_DATASET = "hugginglearners/kvasir-seg"
SPLIT_SEED = 42
TRAIN_RATIO, VAL_RATIO = 0.8, 0.1  # test gets the rest

# Directories
ROOT = Path.cwd().resolve()
OUT_ROOT = ROOT / "out"
IMG_DIR = OUT_ROOT / "images"
MASK_DIR = OUT_ROOT / "masks"
META_DIR = OUT_ROOT / "metadata"
MANIFEST_DIR = OUT_ROOT / "manifests"
for d in [IMG_DIR, MASK_DIR, META_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

META_CSV = META_DIR / "metadata.csv"
MANIFEST_CSV = MANIFEST_DIR / "image_mask_manifest.csv"

print("Root:", ROOT)
print("HF dataset:", HF_DATASET)
print("Output root:", OUT_ROOT)


Root: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep
HF dataset: hugginglearners/kvasir-seg
Output root: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out


In [3]:
# Load dataset from Hugging Face
try:
    ds = load_dataset(HF_DATASET)["train"]
    print("Loaded from HF:", HF_DATASET, ds)
except Exception as e:
    raise RuntimeError(f"HF load failed for {HF_DATASET}: {e}. Try another mirror (e.g., 'voidful/kvasir-seg') or place a local zip with images/masks.")

print("Columns:", ds.column_names)


RuntimeError: HF load failed for hugginglearners/kvasir-seg: Dataset 'hugginglearners/kvasir-seg' doesn't exist on the Hub or cannot be accessed.. Try another mirror (e.g., 'voidful/kvasir-seg') or place a local zip with images/masks.

In [ ]:
# Create splits
n = len(ds)
train_n = int(math.floor(n * TRAIN_RATIO))
val_n = int(math.floor(n * VAL_RATIO))
indices = list(range(n))
random.Random(SPLIT_SEED).shuffle(indices)
train_idx = indices[:train_n]
val_idx = indices[train_n:train_n+val_n]
test_idx = indices[train_n+val_n:]

split_map = {idx: "train" for idx in train_idx}
split_map.update({idx: "validation" for idx in val_idx})
split_map.update({idx: "test" for idx in test_idx})

print({"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)})


In [ ]:
def save_image(img, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(path, format="PNG")


def save_mask(mask, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    mask.save(path, format="PNG")

meta_rows = []
man_rows = []

for idx, ex in tqdm(enumerate(ds), total=len(ds), desc="saving"):
    split = split_map[idx]
    img = ex["image"]  # PIL
    mask = ex.get("mask") or ex.get("segmentation") or ex.get("label")
    if mask is None:
        raise RuntimeError("Mask field not found; check dataset columns")
    img_id = f"{split}_{idx:05d}"
    img_path = IMG_DIR / split / f"{img_id}.png"
    mask_path = MASK_DIR / split / f"{img_id}.png"

    save_image(img, img_path)
    save_mask(mask, mask_path)

    meta_rows.append({
        "split": split,
        "img_id": img_id,
        "image_path": str(img_path),
        "mask_path": str(mask_path),
        "orig_height": getattr(img, "height", None),
        "orig_width": getattr(img, "width", None),
    })
    man_rows.append({
        "split": split,
        "img_id": img_id,
        "image_path": str(img_path),
        "mask_path": str(mask_path),
        "exists_img": img_path.exists(),
        "exists_mask": mask_path.exists(),
    })


In [ ]:
meta = pd.DataFrame(meta_rows)
manifest = pd.DataFrame(man_rows)

meta.to_csv(META_CSV, index=False)
manifest.to_csv(MANIFEST_CSV, index=False)

print("Saved metadata:", META_CSV, "rows", len(meta))
print("Saved manifest:", MANIFEST_CSV, "rows", len(manifest))
print(meta.groupby("split").size())
print(manifest[["exists_img", "exists_mask"]].all())
